# AI Enterprise Knowledge Manager Capstone

**Framework:** OpenAI Agents SDK (`openai-agents`)
**Model Provider:** Groq (OpenAI-compatible endpoint) — used because the OpenAI key has no billing enabled. The SDK is model-agnostic, so pointed every agent's model at Groq via `OpenAIChatCompletionsModel`, using `openai/gpt-oss-120b` (Groq's currently recommended model for reliable tool/function calling — Groq's Llama chat models like `llama-3.3-70b-versatile` are being deprecated and are known to emit malformed tool calls under the OpenAI-style function-calling format).
**Implementation:** Real multi-agent system using `Agent`, `Runner`, `function_tool`, `handoff`, `RunContextWrapper`, and human-in-the-loop tool approval.


# Setup: Dependencies and API Key

In [1]:
!pip install -q openai-agents openai pydantic python-dotenv


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 968.5/968.5 kB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.6/142.6 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.4/223.4 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.6 MB/s eta 0:00:00


In [20]:
import os
from google.colab import userdata
groq_key = userdata.get('GROQ_API_KEY')
os.environ['GROQ_API_KEY'] = groq_key
print("Groq API Key: Loaded")
try:
    openai_key = userdata.get('OPENAI_API_KEY')
    if openai_key:
        os.environ['OPENAI_API_KEY'] = openai_key
        print("OpenAI API Key: Loaded ")
except Exception:
    print("OpenAI API Key: Not set ")


Groq API Key: Loaded
OpenAI API Key: Loaded 


In [3]:
import os
import json
import asyncio
from dataclasses import dataclass, field
from typing import Optional, List, Dict, Any
from datetime import datetime
import time

from pydantic import BaseModel, Field
from openai import AsyncOpenAI

from agents import (
    Agent,
    Runner,
    RunContextWrapper,
    OpenAIChatCompletionsModel,
    function_tool,
    handoff,
    ToolInputGuardrail,
    ToolInputGuardrailData,
    ToolInputGuardrailTripwireTriggered,
    ToolGuardrailFunctionOutput,
    set_tracing_disabled,
)

# We're not using an OpenAI API key, so disable the SDK's default trace export to OpenAI
# (otherwise it just prints a harmless "OPENAI_API_KEY is not set" warning every run).
set_tracing_disabled(True)

print("Imports: Complete")


Imports: Complete


In [4]:
# All agents run on Groq's OpenAI-compatible chat completions endpoint.
groq_client = AsyncOpenAI(
    api_key=os.environ.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
)

def groq_model(model_name: str = "openai/gpt-oss-120b") -> OpenAIChatCompletionsModel:
    """Factory so each agent can get its own model instance bound to Groq."""
    return OpenAIChatCompletionsModel(model=model_name, openai_client=groq_client)

print("Model provider: Groq (openai/gpt-oss-120b) wired through OpenAI Agents SDK")


Model provider: Groq (openai/gpt-oss-120b) wired through OpenAI Agents SDK


## Data Models and Context

In [5]:
@dataclass
class KnowledgeContext:
    user_id: str
    user_name: str
    user_role: str
    department: str
    access_level: str
    knowledge_base: Dict[str, Any] = field(default_factory=dict)
    policy_database: Dict[str, Any] = field(default_factory=dict)
    session_history: List[Dict[str, Any]] = field(default_factory=list)
    current_session_id: str = field(default_factory=lambda: f"session_{int(time.time())}")
    # Human-in-the-loop: recommendations that need sign-off before being finalized
    pending_approvals: List[Dict[str, Any]] = field(default_factory=list)

    def can_access(self, resource_type: str) -> bool:
        access_matrix = {
            "basic": ["general_policies", "procedures"],
            "manager": ["general_policies", "procedures", "team_data"],
            "executive": ["general_policies", "procedures", "team_data", "executive_decisions"]
        }
        allowed = access_matrix.get(self.access_level, [])
        return resource_type in allowed

    def add_to_history(self, event: Dict[str, Any]):
        event['timestamp'] = datetime.now().isoformat()
        event['session_id'] = self.current_session_id
        self.session_history.append(event)

print("Context Model: Defined")


Context Model: Defined


## Structured Output Models

In [7]:
class FindingDetail(BaseModel):
    finding: str = Field(description="Finding text")
    relevance_score: float = Field(description="Relevance 0.0-1.0")
    source: str = Field(description="Source document")

class KnowledgeRecommendation(BaseModel):
    query: str
    summary: str
    findings: List[FindingDetail]
    policy_considerations: List[str]
    recommended_actions: List[str]
    confidence_level: float
    requires_approval: bool
    sources_consulted: List[str]
    reasoning: str

print("Output Models: Defined")


Output Models: Defined


In [8]:
MOCK_KNOWLEDGE_BASE = {
    "doc_remote_work": {
        "title": "Remote Work Policy",
        "content": "Remote work available for eligible employees. Software engineers can work remotely up to 3 days per week. Requires manager approval and VPN access.",
        "category": "hr_policies"
    },
    "doc_leave_policy": {
        "title": "Leave and Time Off Policy",
        "content": "Annual leave: 20 days. Sick leave: 10 days per year. Approval required for leaves over 5 consecutive days. Notice period: 2 weeks.",
        "category": "hr_policies"
    },
    "doc_promotion": {
        "title": "Promotion Criteria",
        "content": "Promotions require: 18+ months in current role, manager recommendation, performance review, training completion. Salary increase: 15-20%.",
        "category": "hr_policies"
    }
}

MOCK_POLICY_DATABASE = {
    "remote_work_eligibility": {
        "rule": "Up to 3 days/week",
        "eligible_roles": ["software engineer", "developer", "designer", "manager"],
        "min_tenure_months": 6,
        "approval_chain": ["direct_manager"]
    },
    "promotion_approval": {
        "rule": "Requires executive sign-off",
        "approval_chain": ["direct_manager", "hr_business_partner", "executive"],
        "min_tenure_months": 18
    },
    "leave_approval": {
        "rule": "Leaves over 5 days need approval",
        "approval_chain": ["direct_manager"],
        "max_days_per_year": 30
    }
}

print("Knowledge Base: Initialized")
print(f"Documents: {len(MOCK_KNOWLEDGE_BASE)}")
print(f"Policies: {len(MOCK_POLICY_DATABASE)}")


Knowledge Base: Initialized
Documents: 3
Policies: 3


## Tools Definition

Each tool is registered with `@function_tool` so agents can actually call it themselves via native
function calling — the model decides when to invoke `search_knowledge_base`, `retrieve_document`, etc.
The shared `KnowledgeContext` is injected via `RunContextWrapper`, so tools read/write the same
session history and access-control state that the agents share.

In [10]:
# Global handle the tools use to read/write the current run's context.
# (The Agents SDK passes a RunContextWrapper[KnowledgeContext] into every tool call.)

@function_tool
def search_knowledge_base(wrapper: RunContextWrapper[KnowledgeContext], query: str) -> Dict[str, Any]:
    """Search the company knowledge base (documents/SOPs) for a text query."""
    context = wrapper.context
    context.add_to_history({"event": "search_initiated", "query": query})
    results = []
    query_lower = query.lower()

    for doc_id, doc in context.knowledge_base.items():
        title_match = query_lower in doc['title'].lower()
        content_match = any(word in doc['content'].lower() for word in query_lower.split())
        if title_match or content_match:
            relevance = 0.9 if title_match else 0.7
            results.append({
                'document_id': doc_id,
                'title': doc['title'],
                'relevance': relevance,
                'preview': doc['content'][:100]
            })
    results.sort(key=lambda x: x['relevance'], reverse=True)
    return {"success": True, "results": results, "count": len(results)}


@function_tool
def retrieve_document(wrapper: RunContextWrapper[KnowledgeContext], document_id: str) -> Dict[str, Any]:
    """Retrieve the full content of a specific document by its document_id."""
    context = wrapper.context
    context.add_to_history({"event": "document_retrieved", "document_id": document_id})
    if document_id not in context.knowledge_base:
        return {"success": False, "error": f"Document {document_id} not found"}
    doc = context.knowledge_base[document_id]
    return {"success": True, "document_id": document_id, "title": doc['title'], "content": doc['content']}


@function_tool
def query_policy_database(wrapper: RunContextWrapper[KnowledgeContext], query_type: str) -> Dict[str, Any]:
    """Look up a specific organizational policy rule by its policy key
    (e.g. 'remote_work_eligibility', 'promotion_approval', 'leave_approval')."""
    context = wrapper.context
    context.add_to_history({"event": "policy_query", "query_type": query_type})
    if query_type not in context.policy_database:
        return {"success": False, "error": f"Policy {query_type} not found"}
    policy = context.policy_database[query_type]
    if "executive" in query_type.lower() and not context.can_access("executive_decisions"):
        return {"success": False, "error": "Access denied"}
    return {"success": True, "policy_type": query_type, "policy": policy}


@function_tool
def analyze_content(content: str, analysis_type: str) -> Dict[str, Any]:
    """Analyze a document's text. analysis_type is either 'extract_requirements' or 'extract_constraints'."""
    if analysis_type == "extract_requirements":
        keywords = ['require', 'must', 'need', 'mandatory']
        requirements = [line.strip() for line in content.split('.') if any(k in line.lower() for k in keywords)]
        return {"success": True, "type": analysis_type, "extracted": requirements}
    elif analysis_type == "extract_constraints":
        keywords = ['cannot', 'must not', 'except', 'limitation']
        constraints = [line.strip() for line in content.split('.') if any(k in line.lower() for k in keywords)]
        return {"success": True, "type": analysis_type, "extracted": constraints}
    return {"success": False, "error": "Unknown analysis type"}


@function_tool
def store_recommendation(wrapper: RunContextWrapper[KnowledgeContext], query: str, recommendation: str) -> Dict[str, Any]:
    """Persist a finalized recommendation for this user/session."""
    import hashlib
    context = wrapper.context
    rec_id = hashlib.md5(f"{context.user_id}{query}{time.time()}".encode()).hexdigest()[:8]
    context.add_to_history({"event": "recommendation_stored", "recommendation_id": rec_id})
    return {"success": True, "recommendation_id": rec_id, "stored_at": datetime.now().isoformat()}


print("Tool 1: search_knowledge_base")
print("Tool 2: retrieve_document")
print("Tool 3: query_policy_database")
print("Tool 4: analyze_content")
print("Tool 5: store_recommendation")


Tool 1: search_knowledge_base
Tool 2: retrieve_document
Tool 3: query_policy_database
Tool 4: analyze_content
Tool 5: store_recommendation


## Human Approval Gate
`store_recommendation` is the tool that finalizes/persists a recommendation. We attach a
**tool input guardrail** to it: any time an agent tries to call `store_recommendation` for a
query that touches a sensitive policy area (promotion, executive decisions, or anything the
Policy Agent flagged as needing sign-off), the guardrail trips and execution pauses for a human
to approve or reject — instead of silently auto-storing it.

This is a genuine approval gate (it can block the tool call)

In [11]:
SENSITIVE_KEYWORDS = ["promotion", "salary", "executive", "termination", "compensation"]

def requires_human_approval(query: str) -> bool:
    q = query.lower()
    return any(k in q for k in SENSITIVE_KEYWORDS)


async def approval_guardrail(data: ToolInputGuardrailData) -> ToolGuardrailFunctionOutput:
    """Blocks store_recommendation for sensitive topics until a human approves."""
    context: KnowledgeContext = data.context.context
    try:
        tool_args = json.loads(data.context.tool_arguments or "{}")
    except (json.JSONDecodeError, TypeError):
        tool_args = {}
    query = tool_args.get("query", "")

    if not requires_human_approval(query):
        return ToolGuardrailFunctionOutput.allow()

    # In a real deployment this would page a human / open a ticket and await their decision.
    # Here we simulate the human approval step synchronously so the notebook stays runnable end to end.
    print(f"\n[HUMAN APPROVAL REQUIRED] Query touches a sensitive policy area: '{query}'")
    decision = input("Approve this recommendation for storage? (yes/no): ").strip().lower()

    context.add_to_history({
        "event": "human_approval_decision",
        "query": query,
        "approved": decision == "yes"
    })

    if decision == "yes":
        print("Approved by human reviewer. Proceeding.\n")
        return ToolGuardrailFunctionOutput.allow()
    else:
        print("Rejected by human reviewer. Recommendation will NOT be stored.\n")
        return ToolGuardrailFunctionOutput.reject_content(
            "Storage blocked: a human reviewer rejected this recommendation."
        )


store_recommendation.tool_input_guardrails = [ToolInputGuardrail(guardrail_function=approval_guardrail)]
print("Human approval guardrail attached to store_recommendation")


Human approval guardrail attached to store_recommendation


## Agents Definition

Six specialized agents built on the real `Agent` class from the OpenAI Agents SDK, each with its
own instructions, tools, and model (Groq). The Coordinator uses `handoffs=[...]` so it can route
control to the Search agent, giving a agent-to-agent handoff

Note: Groq's tool-calling schema validator is stricter than OpenAI's and rejects the SDK's default
zero-argument handoff tool schema (empty `properties`). To stay Groq-compatible we gave the handoff
an explicit, non-empty input schema (`HandoffInput`) via `input_type=` + `on_handoff=`.

In [12]:
class HandoffInput(BaseModel):
    reason: str = Field(description="Why control is being handed off to the Search Agent")

def on_search_handoff(wrapper: RunContextWrapper[KnowledgeContext], input_data: HandoffInput):
    wrapper.context.add_to_history({"event": "handoff", "to": "Search Agent", "reason": input_data.reason})
    print(f"[HANDOFF] Coordinator -> Search Agent | reason: {input_data.reason}")

print("Handoff input schema defined (Groq-compatible, non-empty properties)")


Handoff input schema defined (Groq-compatible, non-empty properties)


In [13]:
search_agent = Agent(
    name="Search Agent",
    instructions=(
        "You are the Knowledge Search agent. Use the search_knowledge_base tool to find documents "
        "relevant to the user's question. In your final answer, explicitly list each result's "
        "document_id (e.g. 'doc_remote_work') alongside its title, so the next agent knows exactly "
        "which document IDs to retrieve."
    ),
    model=groq_model(),
    tools=[search_knowledge_base],
)

document_reader_agent = Agent(
    name="Document Reader Agent",
    instructions=(
        "You are the Document Reader agent. The valid document IDs in the knowledge base are: "
        f"{', '.join(MOCK_KNOWLEDGE_BASE.keys())}. Look at the context above (the Search Agent's "
        "findings) to see which of these IDs are relevant, then call retrieve_document with that "
        "exact document_id, followed by analyze_content on its content. Call each tool at most once "
        "per document. If no specific document seems relevant, briefly say so instead of guessing. "
        "Summarize what you find in plain text."
    ),
    model=groq_model(),
    tools=[retrieve_document, analyze_content],
)

policy_expert_agent = Agent(
    name="Policy Expert Agent",
    instructions=(
        "You are the Policy Expert agent. Use query_policy_database to check the relevant organizational "
        "policy rules (e.g. remote_work_eligibility, promotion_approval, leave_approval) and explain the "
        "eligibility/approval requirements clearly."
    ),
    model=groq_model(),
    tools=[query_policy_database],
)

meeting_memory_agent = Agent(
    name="Meeting Memory Agent",
    instructions=(
        "You are the Meeting Memory agent. You track this session's history and prior context so the "
        "team's decisions stay consistent. Summarize relevant prior context for this query if any exists."
    ),
    model=groq_model(),
    tools=[],
)

recommendation_agent = Agent(
    name="Recommendation Agent",
    instructions=(
        "You are the Recommendation agent. You do NOT have a search tool. Base your answer only on "
        "the information already gathered earlier in this conversation by the other agents (search "
        "results, document analysis, and policy checks). Synthesize that into a clear, actionable "
        "recommendation for the employee, in plain text. When your recommendation is final, call "
        "store_recommendation exactly once to persist it, passing the query and your recommendation text."
    ),
    model=groq_model(),
    tools=[store_recommendation],
)

knowledge_curator_agent = Agent(
    name="Knowledge Curator Agent",
    instructions=(
        "You are the Knowledge Curator agent. Review the recommendation for accuracy, completeness, and "
        "policy alignment. Point out any gaps or risks before it's finalized."
    ),
    model=groq_model(),
    tools=[],
)

coordinator_agent = Agent(
    name="Coordinator Agent",
    instructions=(
        "You are the Coordinator for an Enterprise Knowledge Manager system. Analyze the user's query and "
        "hand off to the Search Agent to begin gathering information. Always start by handing off to search, "
        "and give a short one-sentence reason for the handoff."
    ),
    model=groq_model(),
    handoffs=[handoff(search_agent, input_type=HandoffInput, on_handoff=on_search_handoff)],
)

print("Agent 1: Coordinator (routes via handoff)")
print("Agent 2: Search Agent")
print("Agent 3: Document Reader Agent")
print("Agent 4: Policy Expert Agent")
print("Agent 5: Meeting Memory Agent")
print("Agent 6: Recommendation Agent")
print("Agent 7: Knowledge Curator Agent")


Agent 1: Coordinator (routes via handoff)
Agent 2: Search Agent
Agent 3: Document Reader Agent
Agent 4: Policy Expert Agent
Agent 5: Meeting Memory Agent
Agent 6: Recommendation Agent
Agent 7: Knowledge Curator Agent


## Orchestrator and Workflow

Each stage now runs as a real `Runner.run(...)` call against the Agents SDK, sharing one
`KnowledgeContext` across the whole pipeline (context/memory management). The Coordinator's
handoff to the Search Agent is exercised explicitly first, then the remaining specialists run
in sequence, each able to call its own tools. Each stage's input includes a running summary of
what earlier agents found — so, for example, the Document Reader Agent actually knows which
document_id the Search Agent surfaced instead of guessing blind.

**Error handling:** Groq's hosted open-weight models occasionally emit a malformed/hallucinated
tool call, or spin through tool calls without converging within the turn limit  Each stage runs through `run_stage_with_retry`,
which retries a couple of times and, if it still fails, logs the error to the session history and
continues the pipeline with a clearly-labeled fallback instead of crashing the whole run.

In [14]:
from openai import BadRequestError
from agents import MaxTurnsExceeded

async def run_stage_with_retry(agent: Agent, message: str, context: KnowledgeContext, max_retries: int = 2) -> str:
    """Run one agent step. Groq's smaller/faster models occasionally emit a malformed or
    nonexistent tool call, or spin through tool calls without converging (a known quirk with
    hosted open-weight models, not a bug in our code). Rather than crashing the whole pipeline on
    that, retry a couple of times, then fall back to a clearly-labeled partial result so the rest
    of the workflow can still complete."""
    last_error = None
    for attempt in range(1, max_retries + 2):
        try:
            result = await Runner.run(agent, message, context=context, max_turns=8)
            return result.final_output
        except (BadRequestError, MaxTurnsExceeded) as e:
            last_error = e
            context.add_to_history({
                "event": "agent_error",
                "agent": agent.name,
                "attempt": attempt,
                "error": str(e)[:300],
            })
            print(f"  [WARN] {agent.name} hit an error on attempt {attempt}/{max_retries + 1}: "
                  f"{str(e)[:150]}")
    fallback = f"[{agent.name} could not complete after {max_retries + 1} attempts and was skipped. Last error: {str(last_error)[:150]}]"
    print(f"  [ERROR] {agent.name} giving up after retries — continuing pipeline with a fallback note.")
    context.add_to_history({"event": "agent_fallback", "agent": agent.name})
    return fallback


class Orchestrator:
    def __init__(self):
        self.stages = [
            ("coordinator", coordinator_agent),
            ("search", search_agent),
            ("document_reader", document_reader_agent),
            ("policy_expert", policy_expert_agent),
            ("meeting_memory", meeting_memory_agent),
            ("recommendation", recommendation_agent),
            ("knowledge_curator", knowledge_curator_agent),
        ]

    async def execute(self, user_query: str, context: KnowledgeContext) -> Dict[str, Any]:
        print("WORKFLOW EXECUTION")
        print(f"Query: {user_query}")
        print(f"User: {context.user_name} ({context.user_role})\n")

        results = {}
        # Each stage sees the original query plus everything prior stages found, so e.g. the
        # Document Reader Agent actually knows which document_id the Search Agent surfaced,
        # instead of guessing blind and looping until it hits the turn limit.
        conversation_notes: List[str] = []

        for stage_name, agent in self.stages:
            print(f"Step: {agent.name} running...")
            context.add_to_history({"event": "agent_called", "agent": agent.name})

            if conversation_notes:
                stage_input = (
                    f"User query: {user_query}\n\n"
                    "Context gathered so far by earlier agents:\n" + "\n\n".join(conversation_notes)
                )
            else:
                stage_input = user_query

            output = await run_stage_with_retry(agent, stage_input, context)
            results[stage_name] = output
            conversation_notes.append(f"[{agent.name}]: {output}")
            print("Complete\n")

        # Also run a direct tool call so we have structured search data for the final output object.
        search_results = search_knowledge_base(RunContextWrapper(context=context), user_query) \
            if False else self._direct_search(user_query, context)

        results['structured_output'] = self._create_output(user_query, search_results, results, context)

        print("WORKFLOW COMPLETE")
        return results

    def _direct_search(self, query: str, context: KnowledgeContext) -> Dict[str, Any]:
        # Same matching logic as the search tool, used here just to assemble the structured summary.
        results = []
        query_lower = query.lower()
        for doc_id, doc in context.knowledge_base.items():
            title_match = query_lower in doc['title'].lower()
            content_match = any(word in doc['content'].lower() for word in query_lower.split())
            if title_match or content_match:
                relevance = 0.9 if title_match else 0.7
                results.append({'document_id': doc_id, 'title': doc['title'], 'relevance': relevance})
        results.sort(key=lambda x: x['relevance'], reverse=True)
        return {"success": True, "results": results, "count": len(results)}

    def _create_output(self, query: str, search_results: Dict, results: Dict, context: KnowledgeContext) -> KnowledgeRecommendation:
        findings = [
            FindingDetail(finding=doc['title'], relevance_score=doc['relevance'], source=doc['document_id'])
            for doc in search_results.get('results', [])
        ]

        recommendation_text = str(results.get('recommendation', 'No recommendation'))[:300]

        return KnowledgeRecommendation(
            query=query,
            summary=recommendation_text,
            findings=findings,
            policy_considerations=["Consult manager", "Review policies", "Document decisions"],
            recommended_actions=["Review documentation", "Contact HR", "Follow approval chain"],
            confidence_level=0.92 if findings else 0.5,
            requires_approval=requires_human_approval(query),
            sources_consulted=[doc['document_id'] for doc in search_results.get('results', [])],
            reasoning="Based on organizational policies and knowledge base findings gathered by the agent pipeline."
        )

orchestrator = Orchestrator()
print("Orchestrator: Initialized")


Orchestrator: Initialized


## Demo: Execute System

Run 1 is a routine, non-sensitive query — it flows straight through with no approval needed.

In [15]:
async def run_demo():
    demo_context = KnowledgeContext(
        user_id="EMP_001",
        user_name="Alice Johnson",
        user_role="Software Engineer",
        department="Engineering",
        access_level="manager",
        knowledge_base=MOCK_KNOWLEDGE_BASE,
        policy_database=MOCK_POLICY_DATABASE
    )

    user_query = "Can I work remotely? What are the requirements?"
    results = await orchestrator.execute(user_query, demo_context)

    recommendation = results['structured_output']

    print("STRUCTURED OUTPUT")
    print(f"Query: {recommendation.query}")
    print(f"\nSummary:\n{recommendation.summary}")
    print(f"\nConfidence: {recommendation.confidence_level:.0%}")
    print(f"\nSources:")
    for source in recommendation.sources_consulted:
        print(f"  - {source}")
    print(f"\nFindings:")
    for i, finding in enumerate(recommendation.findings, 1):
        print(f"  {i}. {finding.finding} (Relevance: {finding.relevance_score:.0%})")
    print(f"\nPolicy Considerations:")
    for item in recommendation.policy_considerations:
        print(f"  - {item}")
    print(f"\nRecommended Actions:")
    for item in recommendation.recommended_actions:
        print(f"  - {item}")
    print(f"\nApproval Required: {recommendation.requires_approval}")
    print(f"\nReasoning: {recommendation.reasoning}")

    return demo_context

demo_context = await run_demo()


WORKFLOW EXECUTION
Query: Can I work remotely? What are the requirements?
User: Alice Johnson (Software Engineer)

Step: Coordinator Agent running...
[HANDOFF] Coordinator -> Search Agent | reason: User is asking about remote work eligibility and requirements, needing information gathering.
Complete

Step: Search Agent running...
Complete

Step: Document Reader Agent running...
Complete

Step: Policy Expert Agent running...
Complete

Step: Meeting Memory Agent running...
Complete

Step: Recommendation Agent running...
Complete

Step: Knowledge Curator Agent running...
Complete

WORKFLOW COMPLETE
STRUCTURED OUTPUT
Query: Can I work remotely? What are the requirements?

Summary:
**Recommendation**

You can work remotely if you meet the following conditions and follow the steps below:

### Eligibility
1. **Job title** – Must be one of the eligible roles (Software Engineer, Developer, Designer, Manager, or any other title listed under *eligible_roles* in the Remote Work Polic

Confidence: 

## Demo: Human Approval in Action

Run 2 asks about **promotion** — a sensitive topic — which trips the guardrail on
`store_recommendation` and pauses for a human decision before the recommendation can be stored.

In [16]:
async def run_sensitive_demo():
    demo_context = KnowledgeContext(
        user_id="EMP_002",
        user_name="Ben Carter",
        user_role="Senior Developer",
        department="Engineering",
        access_level="manager",
        knowledge_base=MOCK_KNOWLEDGE_BASE,
        policy_database=MOCK_POLICY_DATABASE
    )

    user_query = "What are the requirements for a promotion and salary increase?"
    results = await orchestrator.execute(user_query, demo_context)
    print("Requires approval flag:", results['structured_output'].requires_approval)
    return demo_context

sensitive_context = await run_sensitive_demo()


WORKFLOW EXECUTION
Query: What are the requirements for a promotion and salary increase?
User: Ben Carter (Senior Developer)

Step: Coordinator Agent running...
[HANDOFF] Coordinator -> Search Agent | reason: User asks for specific policy details on promotion and salary increase requirements, requiring a search for relevant enterprise documents.
Complete

Step: Search Agent running...
Complete

Step: Document Reader Agent running...
Complete

Step: Policy Expert Agent running...
Complete

Step: Meeting Memory Agent running...
Complete

Step: Recommendation Agent running...

[HUMAN APPROVAL REQUIRED] Query touches a sensitive policy area: 'What are the requirements for a promotion and salary increase?'
Approve this recommendation for storage? (yes/no): yes
Approved by human reviewer. Proceeding.

Complete

Step: Knowledge Curator Agent running...
Complete

WORKFLOW COMPLETE
Requires approval flag: True


## Output JSON

In [17]:
async def show_json():
    results = await orchestrator.execute("Can I work remotely?", demo_context)
    rec = results['structured_output']

    output = {
        "query": rec.query,
        "summary": rec.summary,
        "confidence_level": rec.confidence_level,
        "findings": [{
            "finding": f.finding,
            "relevance_score": f.relevance_score,
            "source": f.source
        } for f in rec.findings],
        "policy_considerations": rec.policy_considerations,
        "recommended_actions": rec.recommended_actions,
        "requires_approval": rec.requires_approval,
        "sources_consulted": rec.sources_consulted,
        "reasoning": rec.reasoning
    }

    print(json.dumps(output, indent=2))

await show_json()


WORKFLOW EXECUTION
Query: Can I work remotely?
User: Alice Johnson (Software Engineer)

Step: Coordinator Agent running...
[HANDOFF] Coordinator -> Search Agent | reason: User asks about remote work eligibility, needs policy information
Complete

Step: Search Agent running...
Complete

Step: Document Reader Agent running...
Complete

Step: Policy Expert Agent running...
Complete

Step: Meeting Memory Agent running...
Complete

Step: Recommendation Agent running...
Complete

Step: Knowledge Curator Agent running...
Complete

WORKFLOW COMPLETE
{
  "query": "Can I work remotely?",
  "summary": "You may work remotely only if you meet the company\u2019s Remote Work Policy requirements. Specifically:\n\n1. **Eligibility** \u2013 Your role must be listed as eligible (e.g., Software Engineer, Developer, Designer, Manager, or any other role explicitly named in the policy) and you must have been employed for",
  "confidence_level": 0.92,
  "findings": [
    {
      "finding": "Remote Work Policy

## Session History (Memory/Context Check)

Quick sanity check that the shared `KnowledgeContext` really did accumulate memory across every
agent call and tool call in the pipeline.

In [18]:
print(f"Total events logged this session: {len(demo_context.session_history)}")
for event in demo_context.session_history[-10:]:
    print(event)


Total events logged this session: 25
{'event': 'agent_called', 'agent': 'Search Agent', 'timestamp': '2026-08-08T21:58:37.522138', 'session_id': 'session_1786226102'}
{'event': 'search_initiated', 'query': 'remote work policy', 'timestamp': '2026-08-08T21:58:37.815830', 'session_id': 'session_1786226102'}
{'event': 'agent_called', 'agent': 'Document Reader Agent', 'timestamp': '2026-08-08T21:58:38.355182', 'session_id': 'session_1786226102'}
{'event': 'document_retrieved', 'document_id': 'doc_remote_work', 'timestamp': '2026-08-08T21:58:39.113515', 'session_id': 'session_1786226102'}
{'event': 'agent_called', 'agent': 'Policy Expert Agent', 'timestamp': '2026-08-08T21:58:40.242370', 'session_id': 'session_1786226102'}
{'event': 'policy_query', 'query_type': 'remote_work_eligibility', 'timestamp': '2026-08-08T21:58:40.948790', 'session_id': 'session_1786226102'}
{'event': 'agent_called', 'agent': 'Meeting Memory Agent', 'timestamp': '2026-08-08T21:58:42.482812', 'session_id': 'session_1